In [1]:
import numpy as np
import napari
import zarr
from skimage.exposure import match_histograms
import dask.array as da
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

from ome_zarr.writer import write_multiscales_metadata
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader
from tqdm import tqdm
import mFISHwarp.utils
import mFISHwarp.zarr

import ray

from cucim.skimage.morphology import white_tophat
from cucim.skimage.morphology import ball
import cupy as cp

In [2]:
# read the source
# image path
# '/mnt/ampa02_data01/tmurakami/240425_whole_4color_2nd_M037-3pb/fused/fused.n5'
# '/mnt/ampa02_data01/tmurakami/240417_whole_4color_1st_M037-3pb/fused/fused.n5'
data_path = '/mnt/ampa02_data01/tmurakami/squid/240726_squid_vglut_vacht/squid_all.zarr'
resolution = 0
chunk_size = (1, 256,256,256)
physical_scale = (1.0,2.0,1.3,1.3)
# lazily load the data of the targeted resolution using dask
imgs = mFISHwarp.zarr.omezarr_bdn5_to_dask(data_path,resolution=0)
    
# rechunk for analysis
imgs = imgs.rechunk(chunk_size)

In [3]:
save_zarr_path = '/mnt/ampa02_data01/tmurakami/squid/240726_squid_vglut_vacht/squid_all_top_hat_10.zarr'
downscale_factor_pyramid = (1,2,2,2)
pyramid_level = 5
axes_info = ['c','z','y','x']

## create zarr to save the histogram matched image.
store = zarr.DirectoryStore(save_zarr_path, dimension_separator='/')
root = zarr.group(store=store)

# np.int16 may not be appropriate, but it works for now to reduce the datasize without clipping.
data_zarr = root.create_dataset('0',shape=imgs.shape,chunks=chunk_size,dtype=np.uint16)

# prepare metadata to zarr
datasets = mFISHwarp.zarr.datasets_metadata_generator(physical_scale, downscale_factor=downscale_factor_pyramid, pyramid_level=pyramid_level)

### write metadata for ome-zarr
write_multiscales_metadata(root, datasets=datasets, axes=axes_info)

In [4]:
index_list = list(np.ndindex(*imgs.numblocks))
chunk_info = imgs.chunks

In [5]:
# img_id = ray.put(imgs)

@ray.remote(num_gpus=0.1)
def top_hat_gpu(index, ballsize=10):
    
    footprint = ball(ballsize)
    img = imgs.blocks[index].compute().squeeze()
    img_cp = cp.asarray(img)
    
    res = cp.asnumpy(white_tophat(img_cp,footprint))

    del footprint
    del img_cp
    cp._default_memory_pool.free_all_blocks()
    
    slicer = tuple(slice(sum(i[:j]),sum(i[:j])+i[j]) for i, j in zip(chunk_info,index))

    root['0'][slicer] = res[np.newaxis, ...]

In [6]:
tasks = []
for index in index_list:
    tasks.append(top_hat_gpu.remote(index))
# using ray.get can make the things wait until it done the all process.
results = ray.get(tasks)

2024-08-10 15:04:13,963	INFO services.py:1374 -- View the Ray dashboard at http://127.0.0.1:8265


In [7]:
# make pyramid images
data = da.from_zarr(root['0'])
mFISHwarp.zarr.pyramid_from_dask_to_zarr(data, root, downscale_factor=downscale_factor_pyramid, resolution_start=1, pyramid_level=pyramid_level, chunk=mFISHwarp.utils.chunks_from_dask(data))